In [0]:
from pyspark.sql.functions import *

In [0]:
source_path = "/Volumes/retailnova/bronze/retailnova_source/retailnova_datasets/customers/"
target_table = "retailnova.bronze.customers"

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)

In [0]:
print("Total records found:", df.count())

Total records found: 100000


In [0]:
# Get the last processed timestamp
last_processed = spark.sql("""
    SELECT last_processed_at
    FROM retailnova.bronze.etl_control
    WHERE source_name = 'customers'
""").first()[0]

print("Last processed:", last_processed)

Last processed: 1900-01-01 00:00:00


In [0]:
# Keep only records newer than the watermark
new_customers = df.filter(
    col("updated_at") > last_processed
)

print("New/changed customers:", new_customers.count())


New/changed customers: 100000


In [0]:
# Write new/changed records to Bronze
new_customers.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("Records written to Bronze:", new_customers.count())

Records written to Bronze: 100000


In [0]:
# Calculate the new watermark
new_watermark = spark.sql("""
    SELECT MAX(updated_at)
    FROM retailnova.bronze.customers
""").first()[0]

print("New watermark:", new_watermark)

New watermark: 2026-08-30 23:59:23


In [0]:
# Update the watermark table
spark.sql(f"""
    UPDATE retailnova.bronze.etl_control
    SET last_processed_at = TIMESTAMP('{new_watermark}')
    WHERE source_name = 'customers'
""")

print("Watermark table updated successfully")

Watermark table updated successfully


In [0]:
%sql
SELECT COUNT(*)
FROM retailnova.bronze.customers;

COUNT(*)
100000


In [0]:
%sql
SELECT *
FROM retailnova.bronze.customers
ORDER BY updated_at DESC
LIMIT 10;

customer_id,customer_name,email,phone,city,state,customer_tier,signup_date,updated_at
C002775,Customer 002775,customer002775@retailnova.com,9235592214,Jaipur,Rajasthan,Gold,2024-06-20,2026-08-30T23:59:23.000Z
C052740,Customer 052740,customer052740@retailnova.com,9078905768,Kolkata,West Bengal,Platinum,2023-11-21,2026-08-30T23:59:22.000Z
C096798,Customer 096798,customer096798@retailnova.com,9078768196,Mumbai,Maharashtra,Gold,2024-06-07,2026-08-30T23:58:39.000Z
C068391,Customer 068391,customer068391@retailnova.com,9456624455,Bengaluru,Karnataka,Platinum,2024-09-23,2026-08-30T23:58:36.000Z
C055962,Customer 055962,customer055962@retailnova.com,9213503621,Kochi,Kerala,Gold,2024-03-04,2026-08-30T23:58:28.000Z
C068038,Customer 068038,customer068038@retailnova.com,9186950755,Kochi,Kerala,Silver,2021-05-29,2026-08-30T23:58:17.000Z
C097712,Customer 097712,customer097712@retailnova.com,9418159797,Ahmedabad,Gujarat,Platinum,2022-04-20,2026-08-30T23:57:57.000Z
C068074,Customer 068074,customer068074@retailnova.com,9711724613,Kolkata,West Bengal,Silver,2023-05-06,2026-08-30T23:57:45.000Z
C041561,Customer 041561,customer041561@retailnova.com,9004957220,Bengaluru,Karnataka,Silver,2023-07-29,2026-08-30T23:57:39.000Z
C040259,Customer 040259,customer040259@retailnova.com,9521216168,Jaipur,Rajasthan,Silver,2023-07-11,2026-08-30T23:57:37.000Z


In [0]:
%sql
SELECT *
FROM retailnova.bronze.etl_control;

source_name,last_processed_at
customers,2026-08-30T23:59:23.000Z
